# ResNet50 Transfer Learning Model
### Chest X-Ray Classification: Normal vs Pneumonia vs COVID-19
This notebook fine-tunes a pretrained ResNet50 model for multi-class chest X-ray classification.

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, models
from PIL import Image
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

In [3]:
# Use Apple GPU if available, otherwise CPU
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

# Dataset paths
data_dir = Path('../data/Dataset')
train_dir = data_dir / 'Train_Validation'
test_dir = data_dir / 'Test'
classes = ['COVID', 'Normal', 'Pneumonia']

Using device: mps


In [4]:
# Preprocessing pipeline - standard for ResNet50
transform = transforms.Compose([
    transforms.Resize((224, 224)),        # ResNet expects 224x224
    transforms.ToTensor(),                 # convert to tensor, scale 0-1
    transforms.Normalize(mean=[0.485, 0.456, 0.406],   # ImageNet mean
                        std=[0.229, 0.224, 0.225])      # ImageNet std
])

In [5]:
class XRayDataset(Dataset):
    def __init__(self, data_dir, transform=None):
        self.data_dir = Path(data_dir)
        self.transform = transform
        self.image_paths = []
        self.labels = []
        
        # Collect image paths and labels from each class folder
        for label, cls in enumerate(classes):
            class_dir = self.data_dir / cls
            for img_path in class_dir.glob('*'):
                self.image_paths.append(img_path)
                self.labels.append(label)
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, index):
        img_path = self.image_paths[index]
        image = Image.open(img_path).convert('RGB')
        image = self.transform(image)
        label = self.labels[index]
        return image, label

# Create datasets
full_dataset = XRayDataset(train_dir, transform=transform)
test_dataset = XRayDataset(test_dir, transform=transform)

# Split FIRST before any training
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_split, val_split = random_split(full_dataset, [train_size, val_size])

# Create DataLoaders
train_loader = DataLoader(train_split, batch_size=32, shuffle=True)
val_loader = DataLoader(val_split, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print("Training samples:", train_size)
print("Validation samples:", val_size)
print("Test samples:", len(test_dataset))

Training samples: 2308
Validation samples: 578
Test samples: 341


In [6]:
# Load pretrained ResNet50
model = models.resnet50(weights='IMAGENET1K_V1')

# Freeze all layers
for param in model.parameters():
    param.requires_grad = False

# Replace final layer with 3-class classifier
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 3)

# Move model to device
model = model.to(device)

print("ResNet50 loaded and modified")
print(f"Final layer: {model.fc}")

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /Users/meera/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100.0%


ResNet50 loaded and modified
Final layer: Linear(in_features=2048, out_features=3, bias=True)


In [7]:
# Only train the final layer parameters
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

In [8]:
num_epochs = 10

for epoch in range(num_epochs):
    # Training phase
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    train_loss = running_loss / len(train_loader)
    train_acc = 100 * correct / total
    
    # Validation phase
    model.eval()
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
    
    val_acc = 100 * val_correct / val_total
    print(f"Epoch {epoch+1}/{num_epochs} - Loss: {train_loss:.4f} - Train Acc: {train_acc:.2f}% - Val Acc: {val_acc:.2f}%")

Epoch 1/10 - Loss: 0.7820 - Train Acc: 67.20% - Val Acc: 78.72%
Epoch 2/10 - Loss: 0.5049 - Train Acc: 82.54% - Val Acc: 83.56%
Epoch 3/10 - Loss: 0.4377 - Train Acc: 85.23% - Val Acc: 83.39%
Epoch 4/10 - Loss: 0.4228 - Train Acc: 85.05% - Val Acc: 85.29%
Epoch 5/10 - Loss: 0.3745 - Train Acc: 87.09% - Val Acc: 84.78%
Epoch 6/10 - Loss: 0.3746 - Train Acc: 87.48% - Val Acc: 86.16%
Epoch 7/10 - Loss: 0.3631 - Train Acc: 86.87% - Val Acc: 84.43%
Epoch 8/10 - Loss: 0.3524 - Train Acc: 87.74% - Val Acc: 86.68%
Epoch 9/10 - Loss: 0.3343 - Train Acc: 87.91% - Val Acc: 85.81%
Epoch 10/10 - Loss: 0.3232 - Train Acc: 88.82% - Val Acc: 85.29%


In [9]:
# Save ResNet model
torch.save(model.state_dict(), '../models/resnet50.pth')
print("ResNet50 model saved")

ResNet50 model saved
